# Backstage: Minimale GitHub-Integration

Backstage soll alle Repositories einer GitHub-Organisation durchsuchen und diejenigen automatisch in den Software Catalog aufnehmen, die im Hauptverzeichnis eine Datei namens `catalog-info.yaml` enthalten.

Das Modul ergänzt den Backstage Catalog um den Github Entity Provider, welcher Github-Projekte durchsucht und deren Catalog-Dateien einliest.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn --cwd packages/backend add @backstage/plugin-catalog-backend-module-github

## Backend-Modul registrieren

Das GitHub Entity Provider Modul wird im Backstage-Backend registriert.

Dazu patchen wir die Datei [packages/backend/src/index.ts](../../mybackstage/packages/backend/src/index.ts)


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage
grep -q \
  "plugin-catalog-backend-module-github" packages/backend/src/index.ts \
|| sed -i "/backend.start();/i backend.add(import('@backstage/plugin-catalog-backend-module-github'));" packages/backend/src/index.ts

# Kontrolle
yarn why @backstage/plugin-catalog-backend-module-github

## GitHub Personal Access Token erstellen (optional)

Backstage benötigt einen GitHub Personal Access Token, um über die GitHub-API nach Repositories und `catalog-info.yaml`-Dateien zu suchen.

* Melde dich bei GitHub an und öffne: [https://github.com/settings/tokens](https://github.com/settings/tokens)
* Profilbild → Settings → Developer settings → Personal access tokens → Tokens (classic)
* Token konfigurieren, mindestens
    * Nur öffentliche Repositories: `public_repo`
    * Öffentliche und private Repositories: `repo`
* Klicke danach auf **Generate token**.
* GitHub zeigt den Token nur direkt nach der Erstellung vollständig an. Kopiere ihn deshalb sofort und speichere ihn nicht direkt in einer YAML-Datei.
* Token in [env-platen.py](../../data/env-platen.py) eintragen



In [ ]:
%%bash
source ~/data/env-platen.py
curl --fail --silent --show-error --header "Authorization: Bearer ${GITHUB_TOKEN}" --header "Accept: application/vnd.github+json" https://api.github.com/user

## GitHub Discovery Provider konfigurieren

- verwendet den Token aus `GITHUB_TOKEN`,
- durchsucht alle Repositories der Organisation,
- sucht im jeweiligen Standard-Branch,
- berücksichtigt nur `/catalog-info.yaml`,
- aktualisiert den Catalog alle 30 Minuten.

Da kein Branch-Filter angegeben ist, verwendet Backstage automatisch den Standard-Branch des jeweiligen Repositorys.

Als Gruppe verwenden wir [https://github.com/marcel-cli-org](https://github.com/marcel-cli-org)


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
cat > app-config.github.yaml <<'EOF'
integrations:
  github:
    - host: github.com
      token: ${GITHUB_TOKEN}

catalog:
  providers:
    github:
      production:
        organization: 'marcel-cli-org'
        catalogPath: '/catalog-info.yaml'
        filters:
          repository: '.*'
        schedule:
          frequency:
            minutes: 30
          timeout:
            minutes: 3
EOF


## Konfiguration prüfen

Mit `yarn backstage-cli config:print` wird die zusammengeführte und aufgelöste Backstage-Konfiguration ausgegeben, ohne die Anwendung zu starten.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Github"
export BACKSTAGE_PORT="3001"

source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn backstage-cli config:print --config ~/mybackstage/app-config.yaml --config ~/mybackstage/app-config.test.yaml --config ~/mybackstage/app-config.github.yaml 

## Backstage starten


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Gitlab"
export BACKSTAGE_PORT="3001"

echo "http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn start --config ~/mybackstage/app-config.yaml --config ~/mybackstage/app-config.test.yaml --config ~/mybackstage/app-config.github.yaml 2>&1 | tee /tmp/backstage-github.log

**Links**

- [GitHub Locations](https://backstage.io/docs/integrations/github/locations/)
- [GitHub Discovery](https://backstage.io/docs/integrations/github/discovery/)